# L2 demo: from an empty directory to a tracked run

In L1 an analysis failed to reproduce three ways: an absolute path, an unpinned
split, and an unrecorded library version. Here we rebuild the same analysis the way
it should have been built the first time: a `uv`-managed project with a locked
environment, a pinned interpreter, relative paths, a seeded model, and each run
logged to MLflow.

There are two kinds of cell below. The **scaffold steps** are terminal commands,
shown in fenced blocks; run those in a terminal, not in the notebook. The **Python
cells** define the functions that, in the real project, live in `src/sensorlab/`,
and then run them, so this notebook still executes top to bottom on its own.

> Data: [UCI Air Quality Data Set](https://archive.ics.uci.edu/dataset/360/air+quality),
> the same roadside sensor series from L1, fetched from UCI on first run.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "mlflow": "mlflow",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


## 1. Scaffold the project (in a terminal)

```bash
uv init sensorlab && cd sensorlab
uv add pandas scikit-learn mlflow
```

`uv init` writes `pyproject.toml` and `.python-version`; `uv add` resolves the whole
dependency graph into `uv.lock`. On any other machine, `uv sync` rebuilds this exact
environment. That lockfile, not a `requirements.txt`, is what makes the rebuild
deterministic rather than merely probable.

The layout the project grows into:

```text
sensorlab/
├── pyproject.toml
├── uv.lock
├── .python-version
├── .gitignore          # data/, .venv/, mlflow.db
├── src/sensorlab/      # load, clean, featurize, train
├── data/               # git-ignored: raw data does not go in git
└── tests/
```

## 2. The functions (`src/sensorlab/`)

In the project these live in importable modules; here we define them in the notebook
so the whole thing runs top to bottom. Two of the L1 defects are fixed in the very
first function: the data path is **relative** and its parent is **created** if
missing, so the fetch works on any machine rather than only the author's.

In [ ]:
import io
import hashlib
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path('data/AirQualityUCI.csv')
URL = 'https://archive.ics.uci.edu/static/public/360/air+quality.zip'


def load(path: Path = DATA) -> pd.DataFrame:
    """Fetch (once) and parse the UCI Air Quality CSV.

    Relative path, parent created if missing: the L1 bug, fixed. The export is
    semicolon separated with comma decimals, has two trailing empty columns, and
    codes missing values as -200.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f'fetching {URL}')
        with urllib.request.urlopen(URL) as response:
            payload = response.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            path.write_bytes(archive.read('AirQualityUCI.csv'))
    df = (
        pd.read_csv(path, sep=';', decimal=',')
        .dropna(axis=1, how='all')
        .dropna(how='all')
    )
    df['ts'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'].str.replace('.', ':', regex=False),
        format='%d/%m/%Y %H:%M:%S',
    )
    return df.replace(-200, np.nan)


raw = load()
print(f'{len(raw)} rows, {raw.ts.min().date()} to {raw.ts.max().date()}')

`clean` and `featurize` take a frame and return values, with no hidden global
state, so they can be imported and tested in isolation. We predict the reference CO
measurement from the cheap sensor channels plus temperature and humidity.

In [ ]:
FEATURES = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)',
    'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH',
]
TARGET = 'CO(GT)'


def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Keep rows with all features and the reference present, in time order."""
    return (
        df.dropna(subset=FEATURES + [TARGET])
        .sort_values('ts')
        .reset_index(drop=True)
    )


def featurize(df: pd.DataFrame):
    """Return the feature matrix and target as arrays."""
    return df[FEATURES].to_numpy(), df[TARGET].to_numpy()


clean_df = clean(raw)
X, y = featurize(clean_df)
print(f'{len(clean_df)} usable rows, {X.shape[1]} features')

`train` takes a seed. The split is **temporal** (fit on the earlier 75%, test on
the later 25%), which is the honest protocol for a sensor series, and the model is
seeded so the run is reproducible.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


def train(X, y, seed: int) -> float:
    """Temporal split, seeded model, return R2 on the held-out later period."""
    cut = int(len(X) * 0.75)
    model = RandomForestRegressor(n_estimators=200, random_state=seed)
    model.fit(X[:cut], y[:cut])
    return r2_score(y[cut:], model.predict(X[cut:]))


print(f'R2 (seed 0): {train(X, y, seed=0):.4f}')

## 3. Track each run (MLflow)

Now make each run a fact you can point to. We log the seed, a hash of the data, and
the git commit, alongside the metric, then run the trainer twice with two seeds.
MLflow stores the runs locally in a small SQLite file, with no server to run.

In [ ]:
import subprocess

import mlflow

# MLflow 3 recommends a local database backend over the bare file store.
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('l02-scaffold')


def data_hash(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]


def git_sha() -> str:
    try:
        out = subprocess.check_output(
            ['git', 'rev-parse', '--short', 'HEAD'], text=True, stderr=subprocess.DEVNULL)
        return out.strip()
    except Exception:
        return 'unknown'


for seed in (0, 1):
    with mlflow.start_run(run_name=f'seed-{seed}'):
        r2 = train(X, y, seed=seed)
        mlflow.log_params({
            'seed': seed,
            'model': 'rf200',
            'data_sha': data_hash(DATA),
            'git_sha': git_sha(),
        })
        mlflow.log_metric('r2', r2)
        print(f'seed {seed}: R2 = {r2:.4f}')

### Compare the two runs

Open the UI in a terminal:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db   # then open http://127.0.0.1:5000
```

You will see two runs that differ only by their seed, with slightly different R2. That
small difference is the whole reason to log the seed: without it, neither number is
reconstructible. The data hash and git SHA make the rest of the run reconstructible
too, which is the provenance the L2 notes argue for.

In the real project these functions live in `src/sensorlab/` behind a command-line
entry point, so the whole run is one command:

```bash
uv run python -m sensorlab.train --seed 0
```

That, plus the lockfile and the tracked runs, is assignment **A1**.